# Оценка качества обученной модели CVSSModel

Оценка качества обученной модели CVSSModel на тестовой выборке `test.parquet`.
Используется `final_model.pt` — результат двухэтапного обучения
(Stage 1 на CVSS v3.1, 122 750 записей; Stage 2 на CVSS v4.0, 4 715 записей).
Результаты сохраняются в `reports/test_evaluation.json` и `reports/figures/`.

In [1]:
import pandas as pd
from src.evaluation import Evaluator

evaluator = Evaluator(
    model_path="models/final_model.pt",
    config_path="configs/train.yaml"
)

test_df = pd.read_parquet("data/processed/test.parquet")
test_df = test_df[test_df.cvss_v4_vector.notna()].reset_index(drop=True)
print(f"Test set (v4 only): {len(test_df):,} записей")

C:\Users\Артём\Desktop\diplom\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20836.45it/s]


[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Test set (v4 only): 972 записей


In [2]:
import time
t0 = time.time()
results = evaluator.evaluate(test_df)
print(f"Время оценки: {time.time() - t0:.1f} сек")

evaluator.save_results(results, "reports/test_evaluation.json")

Время оценки: 200.6 сек


{'json': 'reports\\test_evaluation.json',
 'figures': {'AV': 'reports\\figures\\confusion_AV.png',
  'AC': 'reports\\figures\\confusion_AC.png',
  'AT': 'reports\\figures\\confusion_AT.png',
  'PR': 'reports\\figures\\confusion_PR.png',
  'UI': 'reports\\figures\\confusion_UI.png',
  'VC': 'reports\\figures\\confusion_VC.png',
  'VI': 'reports\\figures\\confusion_VI.png',
  'VA': 'reports\\figures\\confusion_VA.png',
  'SC': 'reports\\figures\\confusion_SC.png',
  'SI': 'reports\\figures\\confusion_SI.png',
  'SA': 'reports\\figures\\confusion_SA.png',
  'E': 'reports\\figures\\confusion_E.png',
  'severity': 'reports\\figures\\confusion_severity.png'}}

## Агрегированные метрики

In [3]:
agg = results["aggregated"]
print(f"  Macro-F1 (12 метрик):       {agg['macro_f1']:.4f}")
print(f"  Vector Accuracy:            {agg['vector_accuracy']:.4f}")
print(f"  Метрик правильно/запись:    {agg['metrics_correct_avg']:.2f} / 11")
print(f"  MAE по баллу CVSS:          {agg['score_mae']:.3f}")
print(f"  RMSE по баллу CVSS:         {agg['score_rmse']:.3f}")
print(f"  Severity Accuracy:          {agg['severity_accuracy']:.4f}")
print(f"  Severity ±1 уровень:        {agg['severity_within_one']:.4f}")
print(f"  Записей в тесте:            {agg['samples_evaluated']:,}")

  Macro-F1 (12 метрик):       0.7090
  Vector Accuracy:            0.3992
  Метрик правильно/запись:    9.39 / 11
  MAE по баллу CVSS:          1.172
  RMSE по баллу CVSS:         1.985
  Severity Accuracy:          0.6739
  Severity ±1 уровень:        0.9208
  Записей в тесте:            972


## Per-metric результаты

In [4]:
import pandas as pd
per_metric = results["per_metric"]
df_metrics = pd.DataFrame({
    "Метрика": list(per_metric.keys()),
    "F1 (macro)": [per_metric[m]["f1_macro"] for m in per_metric],
    "Accuracy": [per_metric[m]["accuracy"] for m in per_metric],
    "Precision (macro)": [per_metric[m]["precision_macro"] for m in per_metric],
    "Recall (macro)": [per_metric[m]["recall_macro"] for m in per_metric],
})
df_metrics = df_metrics.round(3)
print(df_metrics.to_string(index=False))

# Также сохраним как CSV для вставки в ВКР
df_metrics.to_csv("reports/per_metric_metrics.csv", index=False)

Метрика  F1 (macro)  Accuracy  Precision (macro)  Recall (macro)
     AV       0.517     0.902              0.613           0.500
     AC       0.748     0.930              0.839           0.702
     AT       0.743     0.884              0.736           0.751
     PR       0.631     0.721              0.703           0.611
     UI       0.641     0.883              0.729           0.612
     VC       0.789     0.794              0.793           0.785
     VI       0.820     0.824              0.818           0.825
     VA       0.795     0.799              0.810           0.791
     SC       0.623     0.870              0.657           0.597
     SI       0.670     0.891              0.729           0.630
     SA       0.656     0.892              0.728           0.610
      E       0.875     0.951              0.872           0.896


## Confusion Matrices

In [5]:
from pathlib import Path
from src.evaluation.confusion_matrices import (
    plot_all_per_metric_matrices,
    plot_severity_confusion_matrix
)

figs_dir = Path("reports/figures/confusion_matrices")
figs_dir.mkdir(parents=True, exist_ok=True)

plot_all_per_metric_matrices(per_metric, figs_dir)
print(f"Сохранено: {len(list(figs_dir.glob('*.png')))} confusion matrices")

# Агрегированная матрица severity
plot_severity_confusion_matrix(
    results["true_severities"],
    results["pred_severities"],
    "reports/figures/severity_confusion_matrix.png"
)

Сохранено: 12 confusion matrices


WindowsPath('reports/figures/severity_confusion_matrix.png')

## Сравнение с результатами Stage 2 (валидация)

In [6]:
# Метрики на val из последней эпохи обучения
# (вписаны вручную из лога Stage 2):
val_metrics_stage2 = {
    "AV": 0.534, "AC": 0.724, "AT": 0.744, "PR": 0.629,
    "UI": 0.629, "VC": 0.805, "VI": 0.813, "VA": 0.823,
    "SC": 0.648, "SI": 0.635, "SA": 0.660, "E": 0.881,
}

comparison = pd.DataFrame({
    "Метрика": list(val_metrics_stage2.keys()),
    "F1 (val)":  list(val_metrics_stage2.values()),
    "F1 (test)": [per_metric[m]["f1_macro"] for m in val_metrics_stage2.keys()],
})
comparison["Δ"] = (comparison["F1 (test)"] - comparison["F1 (val)"]).round(3)
print(comparison.round(3).to_string(index=False))

Метрика  F1 (val)  F1 (test)      Δ
     AV     0.534      0.517 -0.017
     AC     0.724      0.748  0.024
     AT     0.744      0.743 -0.001
     PR     0.629      0.631  0.002
     UI     0.629      0.641  0.012
     VC     0.805      0.789 -0.016
     VI     0.813      0.820  0.007
     VA     0.823      0.795 -0.028
     SC     0.648      0.623 -0.025
     SI     0.635      0.670  0.035
     SA     0.660      0.656 -0.004
      E     0.881      0.875 -0.006


## Примеры предсказаний

In [7]:
samples = results["predictions_sample"]
for i, sample in enumerate(samples[:10], 1):
    print(f"\n=== Пример {i} (CVE: {sample.get('cve_id', 'unknown')}) ===")
    print(f"Описание: {sample['description'][:150]}...")
    print(f"True vector:  {sample['true_vector']}")
    print(f"Pred vector:  {sample['pred_vector']}")
    print(f"True score: {sample['true_score']:.1f} ({sample['true_severity']})")
    print(f"Pred score: {sample['pred_score']:.1f} ({sample['pred_severity']})")


=== Пример 1 (CVE: CVE-2025-5851) ===
Описание: Уязвимость функции fromadvsetlanip() (/goform/AdvSetLanip) микропрограммного обеспечения маршрутизаторов Tenda AC15 связана с копированием буфера без ...
True vector:  CVSS:4.0/AV:N/AC:L/AT:N/PR:L/UI:N/VC:H/VI:H/VA:H/SC:N/SI:N/SA:N
Pred vector:  CVSS:4.0/AV:N/AC:L/AT:N/PR:L/UI:N/VC:H/VI:H/VA:H/SC:N/SI:N/SA:N/E:P
True score: 8.7 (High)
Pred score: 7.4 (High)

=== Пример 2 (CVE: CVE-2024-10387) ===
Описание: Уязвимость компонента FactoryTalk платформы для централизованного управления приложениями Rockwell Automation ThinManage связана с возможностью чтения...
True vector:  CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:N/VI:N/VA:H/SC:N/SI:N/SA:N/E:A
Pred vector:  CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:H/VI:H/VA:H/SC:N/SI:N/SA:N/E:A
True score: 8.7 (High)
Pred score: 9.3 (Critical)

=== Пример 3 (CVE: CVE-2024-6938) ===
Описание: A vulnerability has been found in SiYuan 3.1.0 and classified as problematic. Affected by this vulnerability is an unknown f

## Резюме

In [8]:
print("=" * 60)
print("ИТОГОВЫЕ МЕТРИКИ ДЛЯ ВКР")
print("=" * 60)
print(f"Macro-F1 на test set:        {agg['macro_f1']:.4f}")
print(f"Vector Accuracy:             {agg['vector_accuracy']:.4f}")
print(f"MAE по CVSS Score:           {agg['score_mae']:.3f}")
print(f"Severity Accuracy:           {agg['severity_accuracy']:.4f}")
print(f"Severity ±1 уровень:         {agg['severity_within_one']:.4f}")
print()
print(f"Размер тестовой выборки:     {agg['samples_evaluated']:,} записей CVSS v4.0")
print(f"Источник данных:             NVD + БДУ ФСТЭК")

ИТОГОВЫЕ МЕТРИКИ ДЛЯ ВКР
Macro-F1 на test set:        0.7090
Vector Accuracy:             0.3992
MAE по CVSS Score:           1.172
Severity Accuracy:           0.6739
Severity ±1 уровень:         0.9208

Размер тестовой выборки:     972 записей CVSS v4.0
Источник данных:             NVD + БДУ ФСТЭК
